___
# Figure 5b – *Process* Ambulatory Pain Decoding Analysis  
Shirvalkar et al., Nature Medicine 2026

This script performs a frequency-resolved decoding analysis of chronic pain using 
neural spectral power recorded from the **P1's right anterior cingulate cortex**. 
The goal is to quantify how well spectral power at each frequency predicts pain severity, 
measured via MPQ scores.

Inputs:
- `main_dir`: Root directory containing the repository and data folders  
- `P1_neural_spectra.xlsx`: neural spectral power (rows = timepoints/surveys, columns = frequency in Hz)  
- `P1_pain_surveys.xlsx`:  Pain survey data containing MPQ scores  

Output:
- `P1_pain_decoding_AUC.xlsx`: AUCs (columns = frequency bins (Hz))
___

### Repository root (resolved automatically; set `NATMED_REPO` to override)

In [5]:
import os
# Repository root: defaults to the parent of python_code/, i.e. this notebook's
# grandparent folder. Override by setting the NATMED_REPO environment variable
# or replacing this line with an explicit path to your local copy.
main_dir = os.environ.get('NATMED_REPO',
    os.path.abspath(os.path.join(os.getcwd(), os.pardir))) + '/'


In [ ]:
# import modules
import numpy as np
import pandas as pd
from pathlib import Path
import sys

sys.path.append(str(Path(main_dir) / 'python_code/shared_utilities'))
from utility import SignalProcessingUtility as SPU

# LOAD DATA
data_dir       = Path(main_dir) / "data/figure5_processed_ambulatory_decoding"
# load neural spectra (surveys by frequency)
neural_spectra = pd.read_excel(data_dir / "P1_neural_spectra.xlsx", index_col=0)
neural_spectra.columns.name = "Frequency_Hz"
# load pain surveys
pain_surveys   = pd.read_excel(data_dir / "P1_pain_surveys.xlsx")
MPQ            = pain_surveys["MPQ"].values

# =============================================================================
# COMPUTE AUC for linear discriminant analysis of decoding chronic pain from spectral power
# =============================================================================
pain_decoding_AUC = SPU.sliding_frequency_auc(neural_spectra, MPQ)

# =============================================================================
# SAVE AUC to .xlsx file
# =============================================================================
pain_decoding_AUC_df       = pd.DataFrame(pain_decoding_AUC.reshape(1, -1), columns=neural_spectra.columns)
pain_decoding_AUC_df.index = ["AUC"]
pain_decoding_AUC_df.to_excel(data_dir / "P1_pain_decoding_AUC.xlsx")

print("AUC saved to:", data_dir / "P1_pain_decoding_AUC.xlsx")